# GaussianSplat — train on Colab

Upload the `session-<id>.zip` produced by the app, train a 3D Gaussian Splatting scene on Colab's GPU, then download the resulting `output.splat` and load it back into the viewer.

**Runtime:** before running, set **Runtime → Change runtime type → T4 GPU** (free tier is enough).

In [ ]:
# 1. Sanity-check GPU
!nvidia-smi

In [ ]:
# 2. Install nerfstudio + gsplat. ~5 min on a cold runtime.
%pip install --quiet nerfstudio gsplat

In [ ]:
# 3. Upload your session-<id>.zip from the phone (AirDropped to your Mac first).
from google.colab import files
import os, zipfile, shutil

uploaded = files.upload()
if not uploaded:
    raise SystemExit('No file uploaded.')
zip_name = next(iter(uploaded))
shutil.rmtree('session', ignore_errors=True)
os.makedirs('session', exist_ok=True)
with zipfile.ZipFile(zip_name) as z:
    z.extractall('session')
print('contents:', sorted(os.listdir('session'))[:5], '…')

In [ ]:
# 4. Train splatfacto. Drop max-num-iterations to taste (5k = quick demo, 30k = paper-quality).
!ns-train splatfacto \
  --data session \
  --output-dir runs \
  --viewer.quit-on-train-completion True \
  --max-num-iterations 7000

In [ ]:
# 5. Find the trained .ply, convert it to .splat with the repo's helper.
import glob, subprocess, urllib.request, os

plys = sorted(glob.glob('runs/**/point_cloud.ply', recursive=True))
if not plys:
    raise SystemExit('Training did not produce a point_cloud.ply')
ply = plys[-1]
print('using ply:', ply)

urllib.request.urlretrieve(
    'https://raw.githubusercontent.com/ArioMoniri/GaussianSplat/main/scripts/ply_to_splat.py',
    'ply_to_splat.py',
)
subprocess.check_call(['python', 'ply_to_splat.py', ply, 'output.splat'])
print('wrote output.splat', os.path.getsize('output.splat'), 'bytes')

In [ ]:
# 6. Download to your Mac. AirDrop the file back to the phone, then
# in the app: Train → Import trained splat.
from google.colab import files
files.download('output.splat')